# DSA210 Milestone 1: Data Collection, EDA, and Hypothesis Testing

## Project Topic
This project investigates the relationship between coffee consumption and lifestyle factors such as sleep quality, stress level, and daily habits.

## Goal
The goal of this notebook is to explore the dataset, assess its quality, visualize important patterns, and test hypotheses about the relationship between coffee consumption and sleep/lifestyle variables.

## 1) Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, mannwhitneyu

df = pd.read_excel("DSA210 (Responses).xlsx")
df.head()

## 2) Rename the Columns

In [ ]:
df.columns = [
    "timestamp",
    "email",
    "age",
    "gender",
    "coffee",
    "coffee_type",
    "sleep_hours",
    "sleep_quality",
    "stress",
    "screen_time",
    "exercise"
]

# Drop columns that are not useful for analysis
# 'email' contains personal data (privacy concern) and 'timestamp' is not needed
df.drop(columns=["email", "timestamp"], inplace=True)

df.head()

## 3) Data Overview

In [ ]:
print("Shape:", df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

Each row represents one participant response. Each column represents a variable collected from the survey. The dataset contains **106 participants** and **9 variables** (after removing email and timestamp).

## 4) Attribute Types

The variables in this dataset can be grouped as follows:

- **Categorical (Nominal)**: `gender`, `coffee_type`, `exercise`  
- **Ordinal**: `stress` (1–5 scale), `sleep_quality` (1–5 scale)  
- **Numerical (Discrete)**: `age`, `coffee` (cups per day)  
- **Numerical (Continuous)**: `sleep_hours`, `screen_time`

## 5) Data Quality Check

Before exploring the data, we check for missing values, duplicates, and potential outliers.

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Check for potential outliers in screen_time
# max=20 in describe() is suspicious — let's inspect high values
print("screen_time > 8 hours (possible outliers):")
print(df[df["screen_time"] > 8][["age", "coffee", "screen_time", "stress"]])

The `screen_time` variable has a maximum of 20 hours, which is an extreme value. These rows are kept in the analysis but noted as potential outliers. No missing values were found in the dataset.

## 6) Exploratory Data Analysis

### 6.1 Distribution of Numerical Variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Distribution of Numerical Variables", fontsize=14)

numerical_cols = ["age", "coffee", "sleep_hours", "sleep_quality", "stress", "screen_time"]
xlabels = ["Age", "Cups per day", "Hours", "Rating (1-5)", "Rating (1-5)", "Hours"]

for ax, col, xlabel in zip(axes.flatten(), numerical_cols, xlabels):
    ax.hist(df[col], bins=10, edgecolor='black', color='steelblue', alpha=0.7)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='orange', linestyle=':', label=f'Median: {median_val:.1f}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

**Observations:**
- **Age**: Distribution is right-skewed; most participants are young (around 21-23), but there are a few older participants (50+). Median is a better central measure here.
- **Coffee**: Most participants drink 1-2 cups/day. The distribution is slightly right-skewed.
- **Sleep hours**: Approximately normally distributed, centered around 7 hours.
- **Sleep quality & Stress**: Both rated on a 1-5 scale. Stress is slightly left-skewed (higher stress is common).
- **Screen time**: Heavily right-skewed with a clear outlier at 20 hours.

### 6.2 Distribution of Categorical Variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Distribution of Categorical Variables", fontsize=14)

categorical_cols = ["gender", "coffee_type", "exercise"]

for ax, col in zip(axes, categorical_cols):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=15)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.5, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

**Observations:**
- **Gender**: The sample is not balanced — there are more female participants than male. This may affect generalizability.
- **Coffee type**: Turkish coffee and other types are the most common. Latte and Americano are less frequent.
- **Exercise**: The majority of participants do not exercise regularly.

### 6.3 Box Plots: Spread and Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Box Plots of Key Numerical Variables", fontsize=14)

for ax, col in zip(axes, ["sleep_hours", "stress", "screen_time"]):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel("Value")
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    ax.set_xlabel(f'IQR={iqr:.1f}, Median={df[col].median():.1f}')

plt.tight_layout()
plt.show()

Box plots reveal that `screen_time` has a visible outlier (20 hours). `sleep_hours` is well-behaved with most values between 6 and 8 hours. `stress` scores are concentrated between 3 and 5.

### 6.4 Bivariate Analysis: Coffee vs. Key Variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Coffee Consumption vs. Lifestyle Variables", fontsize=14)

# Coffee vs Sleep Hours (scatter)
axes[0].scatter(df["coffee"], df["sleep_hours"], alpha=0.5, color='steelblue')
m, b = np.polyfit(df["coffee"], df["sleep_hours"], 1)
x_line = np.linspace(df["coffee"].min(), df["coffee"].max(), 100)
axes[0].plot(x_line, m*x_line + b, color='red', linewidth=2, label=f'y={m:.2f}x+{b:.2f}')
axes[0].set_xlabel("Cups of Coffee per Day")
axes[0].set_ylabel("Sleep Hours")
axes[0].set_title("Coffee vs Sleep Hours")
axes[0].legend()

# Coffee vs Stress (box per category)
sns.boxplot(x="coffee", y="stress", data=df, ax=axes[1], palette="Blues")
axes[1].set_title("Coffee vs Stress Level")
axes[1].set_xlabel("Cups of Coffee per Day")
axes[1].set_ylabel("Stress Level (1-5)")

# Coffee vs Sleep Quality (box per category)
sns.boxplot(x="coffee", y="sleep_quality", data=df, ax=axes[2], palette="Greens")
axes[2].set_title("Coffee vs Sleep Quality")
axes[2].set_xlabel("Cups of Coffee per Day")
axes[2].set_ylabel("Sleep Quality (1-5)")

plt.tight_layout()
plt.show()

The scatter plot shows a slight negative trend between coffee consumption and sleep hours. The regression line suggests that each additional cup of coffee is associated with a small decrease in sleep duration. The relationship between coffee and stress or sleep quality appears weaker visually.

### 6.5 Group Comparisons: Exercise and Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Group Comparisons", fontsize=14)

# Exercise vs sleep hours
sns.boxplot(x="exercise", y="sleep_hours", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Exercise vs Sleep Hours")
axes[0].set_xlabel("Exercises Regularly?")
axes[0].set_ylabel("Sleep Hours")

# Gender vs coffee
sns.boxplot(x="gender", y="coffee", data=df, ax=axes[1], palette="Set1")
axes[1].set_title("Gender vs Coffee Consumption")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Cups of Coffee per Day")

plt.tight_layout()
plt.show()

Participants who exercise regularly appear to sleep slightly more on average. Coffee consumption seems similar across genders, though there is some variation.

### 6.6 Correlation Matrix and Heatmap

In [ ]:
corr_matrix = df.corr(numeric_only=True)
print(corr_matrix)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()

**Key correlations from the heatmap:**
- `coffee` and `sleep_hours`: r = -0.24 (negative, moderate-weak) — more coffee is associated with less sleep.
- `stress` and `sleep_quality`: r = -0.21 (negative) — higher stress is associated with lower sleep quality.
- `stress` and `screen_time`: r = 0.24 (positive) — more screen time before bed is associated with higher stress.
- Most other correlations are weak (|r| < 0.15).

## 7) Hypothesis Testing

Based on the EDA, we formulate and test three hypotheses related to coffee consumption and lifestyle variables.

---

### Hypothesis 1: Coffee and Sleep Hours

**H₀**: There is no linear correlation between daily coffee consumption and sleep hours.  
**H₁**: There is a significant negative correlation between daily coffee consumption and sleep hours.  
**Test**: Pearson's correlation coefficient (appropriate since both variables are numerical/continuous).  
**Significance level**: α = 0.05

In [ ]:
corr, p_value = pearsonr(df["coffee"], df["sleep_hours"])

print("=== Hypothesis 1: Coffee vs Sleep Hours ===")
print(f"Pearson r  : {corr:.4f}")
print(f"p-value    : {p_value:.4f}")
print()
if p_value < 0.05:
    print(f"Result: REJECT H₀ (p={p_value:.4f} < 0.05)")
    print("There is a statistically significant negative correlation between coffee and sleep hours.")
else:
    print(f"Result: FAIL TO REJECT H₀ (p={p_value:.4f} >= 0.05)")

---

### Hypothesis 2: Coffee and Stress Level

**H₀**: There is no monotonic relationship between coffee consumption and stress level.  
**H₁**: There is a significant monotonic relationship between coffee consumption and stress level.  
**Test**: Spearman's rank correlation (stress is ordinal, so Spearman is more appropriate than Pearson).  
**Significance level**: α = 0.05

In [ ]:
rho, p_spearman = spearmanr(df["coffee"], df["stress"])

print("=== Hypothesis 2: Coffee vs Stress ===")
print(f"Spearman rho: {rho:.4f}")
print(f"p-value      : {p_spearman:.4f}")
print()
if p_spearman < 0.05:
    print(f"Result: REJECT H₀ (p={p_spearman:.4f} < 0.05)")
    print("There is a statistically significant monotonic relationship between coffee and stress.")
else:
    print(f"Result: FAIL TO REJECT H₀ (p={p_spearman:.4f} >= 0.05)")
    print("No significant monotonic relationship found between coffee consumption and stress level.")

---

### Hypothesis 3: Exercise and Sleep Hours

**H₀**: People who exercise regularly and those who do not have the same median sleep hours.  
**H₁**: People who exercise regularly sleep significantly more than those who do not.  
**Test**: Mann-Whitney U test (non-parametric test for comparing two independent groups; appropriate because normality is not guaranteed for a small-n split).  
**Significance level**: α = 0.05

In [ ]:
exercisers = df[df["exercise"] == "Yes"]["sleep_hours"]
non_exercisers = df[df["exercise"] == "No"]["sleep_hours"]

u_stat, p_mw = mannwhitneyu(exercisers, non_exercisers, alternative="greater")

print("=== Hypothesis 3: Exercise vs Sleep Hours ===")
print(f"Exercisers median sleep    : {exercisers.median():.2f} hrs (n={len(exercisers)})")
print(f"Non-exercisers median sleep: {non_exercisers.median():.2f} hrs (n={len(non_exercisers)})")
print(f"Mann-Whitney U: {u_stat:.1f}")
print(f"p-value       : {p_mw:.4f}")
print()
if p_mw < 0.05:
    print(f"Result: REJECT H₀ (p={p_mw:.4f} < 0.05)")
    print("Exercisers sleep significantly more than non-exercisers.")
else:
    print(f"Result: FAIL TO REJECT H₀ (p={p_mw:.4f} >= 0.05)")
    print("No significant difference in sleep hours between exercisers and non-exercisers.")

## 8) Summary of Findings

| Hypothesis | Test Used | Result | Interpretation |
|---|---|---|---|
| Coffee ↔ Sleep Hours | Pearson r | Significant (p<0.05) | More coffee → less sleep |
| Coffee ↔ Stress | Spearman ρ | See output | Ordinal test for stress scale |
| Exercise ↔ Sleep Hours | Mann-Whitney U | See output | Non-parametric group comparison |

**Key takeaways:**
1. Coffee consumption has a statistically significant, moderate-weak negative correlation with sleep hours (r ≈ -0.24, p ≈ 0.014).
2. Stress and screen time are positively correlated (r ≈ 0.24), suggesting that more screen time before bed is associated with higher stress.
3. The sample is predominantly young (median age ≈ 22) and female, which may limit the generalizability of findings.
4. The `screen_time` outlier (20 hours) should be flagged and potentially excluded in further analysis.

**Limitations:**
- Survey data is self-reported and subject to recall bias.
- The sample size (n=106) is moderate but not large enough to detect small effects reliably.
- The sample is not representative of the general population (mostly young students).
- Correlation does not imply causation.